In [22]:
import gurobipy as gp
from gurobipy import GRB
import random
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import time
import json
import ast
from pathlib import Path

In [5]:
#solvs the apglp using gurobi mip with input graph G, initial value a and constant difference d
#returns the assigned edge weights and time used for processing
#gurobi_mip (NetworkX graph object: G, int: a, int: d)
def gurobi_mip (G, a, d):

    print(f"running: a={a}; b={d}")
    
    #list of vertex lables Y
    vertex_labels = []
    n=G.order()
    for i in range(n):
        vertex_labels.append(a + i*d)

    #record processing time
    start_time = time.process_time()

    m = gp.Model()

    #for model description and log information, comment out the following line
    m.Params.OutputFlag = 0
    
    # add binary variables: t[v, w] == 1 if vertex v is assigned label w
    t = m.addVars(G.nodes, vertex_labels, vtype=GRB.BINARY, name="t")
    
    # add variables for edge labels: x[e] for each edge e, with x[e] >= 1 and <=a+(n-2)*d (the second highest vertex label
    x = m.addVars(G.edges, lb=1, ub=a+(n-2)*d, vtype=GRB.INTEGER, name="x")
    
    #each vertex gets exactly one label
    for v in G.nodes:
        m.addConstr(gp.quicksum(t[v, w] for w in vertex_labels) == 1,
                name=f"one_w_for_v{v}")
    
    #each label is assigned to exactly one vertex
    for w in vertex_labels:
        m.addConstr(gp.quicksum(t[v, w] for v in G.nodes) == 1,
                name=f"one_v_for_w{w}")
    
    #for each vertex, the sum of its incident edge labels equals the vertex label.
    for v in G.nodes:
        #incident edges of vertex v.
        incident_edges = [e for e in G.edges if v in e]
        m.addConstr(gp.quicksum(x[e] for e in incident_edges) == gp.quicksum(w * t[v, w] for w in vertex_labels), name=f"balance_v{v}")
    
    #set a dummy objective (minimize 0) since we only require one feasible solution
    m.setObjective(0, GRB.MINIMIZE)

    m.reset()
    #run the optimizer to solve the model
    m.optimize()

    #check if an optimal solution was found
    if m.status == GRB.OPTIMAL: 

        #store time in microseconds
        elapsed_time = (time.process_time() - start_time)*(10**6)
        
        #check which label is assigned to each vertex and store it in dic node_weights in format {vertex : weight}
        node_weights={}
        for v in G.nodes:
            for w in vertex_labels:
                if t[v, w].X == 1:  
                    node_weights[v] = w
                               
        edge_weights = {e:int(x[e].X) for e in G.edges}
        
        #draw the graph using NetworkX drawing functions
        pos = nx.spring_layout(G,seed=2)
        plt.figure(figsize=(6, 6))
        nx.draw(G, pos, labels=node_weights, with_labels=True, node_size=500, font_size=12, node_color='white', edgecolors='black', linewidths=3, width=3)
        nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_weights, font_size=10)
        plt.show()
        
        return node_weights, edge_weights, m.Runtime*10**6, m.NumVars, m.NumConstrs, m.NumNZs
    return 0, 0, 0, 0, 0, 0


    

In [37]:
# Solves the APGLP using Gurobi MIP for an input graph G with given vertex label
def gurobi_fixed(G,a,d):

    # Record processing time
    start_time = time.process_time()

    m = gp.Model()
    
    #for model description and log information, comment out the following line
    m.Params.OutputFlag = 0

    # Retrieve fixed vertex labels from the graph
    node_weights = nx.get_node_attributes(G, 'weight')

    # Compute upper bounds for each edge
    edge_ubs = {
        e: min(G.nodes[e[0]]['weight'], G.nodes[e[1]]['weight']) for e in G.edges
    }

    # add variables for edge labels: x[e] for each edge e
    x = m.addVars(G.edges, lb=1, ub=a+(n-2)*d, vtype=GRB.INTEGER, name="x")

    #for each vertex, the sum of its incident edge labels equals the vertex label.
    for v in G.nodes:
        #incident edges of vertex v.
        incident_edges = [e for e in G.edges if v in e]
        m.addConstr(gp.quicksum(x[e] for e in incident_edges) == node_weights[v], name=f"balance_v{v}")

    #set a dummy objective (minimize 0) since we only require one feasible solution
    m.setObjective(0, GRB.MINIMIZE)

    # Solve the MIP model
    m.reset()
    m.optimize()

    if m.status == GRB.OPTIMAL:
        # Compute elapsed time (in microseconds)
        elapsed_time = (time.process_time() - start_time) * (10**6)
        
        #store the edge weights as a dictionary
        edge_weights = {e: int(x[e].X) for e in G.edges}
        
        #draw the graph using NetworkX drawing functions
        pos = nx.spring_layout(G,seed=2)
        plt.figure(figsize=(6, 6))
        nx.draw(G, pos, labels=node_weights, with_labels=True, node_size=500, font_size=12, node_color='white', edgecolors='black', linewidths=3, width=3)
        nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_weights, font_size=10)
        plt.show()

        return edge_weights, m.Runtime*10**6, m.NumVars, m.NumConstrs, m.NumNZs
    
    # Return a tuple of zeros if no optimal solution is found.
    return 0, 0, 0, 0, 0


In [74]:
# Takes a .csv file path and an output file path as inputs.
# For each graph in the input file, generates all valid APGs based on the given starting value and constant difference.
# Records the runtime and writes the results to the output file.
def mip_find_apgs(file_name, mode):

    df = pd.read_csv(file_name)

    #define the column names for a DataFrame df
    col_name = ['APG_label_availability','Vertex_labels', 'Edge_labels', 'Gurobi_runtime/microseconds', 'Gurobi_num_vars', 'Gurobi_num_constrs', 'Gurobi_nonzero_coefs']
    dict = {col_name[0]:[],
            col_name[1]:[],
            col_name[2]:[],
            col_name[3]:[],
            col_name[4]:[],
            col_name[5]:[],
            col_name[6]:[],
           }
    #create the DataFrame df with column names defined
    df_g = pd.DataFrame(dict)

    #find labeling for each graph with given a and d
    n=df.shape[0]
    for i in range(n):
        a = int(df['a'][i])
        d = int(df['d'][i])

        # Build a NetworkX graph using the provided adjacency list.        
        ad_str = df['Adjacency_list'][i]
        ad_dict=json.loads(ad_str)
        ad_list = pd.DataFrame(ad_dict)
        G = nx.from_pandas_adjacency(ad_list)

        #for unconstrained APGLP
        if mode == 'N':
            vertex_labels, edge_labels, gurobi_runtime, num_vars, num_constrs, nonzero_coefs = gurobi_mip(G, a, d)

        #for fixed vertex-labed APGLP
        else:
            #assign node weight to G
            print(df.columns.tolist())
            vlabels = df['Vertex_labels'][i]
            vertex_labels = ast.literal_eval(vlabels)
            nx.set_node_attributes(G, vertex_labels, name='weight')

            #run gurobi_fixed
            edge_labels, gurobi_runtime, num_vars, num_constrs, nonzero_coefs = gurobi_fixed(G,a,d)

        if vertex_labels != 0:
            df2 = {col_name[0]: "F",
            col_name[1]: vertex_labels, col_name[2]: edge_labels, col_name[3]: gurobi_runtime, col_name[4]: num_vars, col_name[5]: num_constrs, col_name[6]: nonzero_coefs}
            df_g = pd.concat([df_g, pd.DataFrame([df2])], ignore_index=True)

    if mode == "F":
        #for node 6->8, [5:], otherwise [4:]
        df = df.drop(df.columns[5:], axis=1)
        p = Path(file_name)
        file_name = p.with_name(p.stem + '_fixed' + p.suffix)
    
    df = pd.concat([df, df_g], axis=1)    
    df.to_csv(file_name, index=False)

    

In [ ]:
# Runs the mip_find_apgs function with the given input and output file paths.
# 'N' for unconstrained APGLP, 'F' for fixed vertex-labeled APGLP.
mip_find_apgs('apgs/base_apgs/apg8_base.csv','F')